# Mapping the Nodes of Universal Nouns Through Recursive Computational Verbs - Executable Visual Notebook

This notebook operationalizes the paper **_Mapping the Nodes of Universal Nouns Through Recursive Computational Verbs_** as a reproducible, presentation-ready research notebook.

## Notebook goals
1. Organize the paper's core maps and cross-domain correspondences into machine-readable tables.
2. Verify selected SHA-256 structural claims directly from executable code.
3. Generate interactive visuals for the paper's main architectures, attractors, and phase structures.
4. Export reusable datasets and standalone HTML figures for downstream analysis.

## Scope conventions
- **Paper-stated** quantities are transcribed from the source paper and labeled as such.
- **Notebook-derived** quantities are computed in this notebook from executable code.
- Visuals are descriptive unless explicitly labeled as a verification or audit figure.

## Source
- PDF: `Mapping the Nodes of Universal Nouns Through Recursive Computational Verbs.pdf`
- Source pages used heavily in this notebook:
  - pages 3-5: Universal Component Map
  - pages 6-8: SHA-256 bridge separation, Sziklai coupling, and dual-wave ontology
  - pages 9-10: Sarrus Isomorphism cross-substrate table
  - pages 11-16: stroboscopic 33 Hz primitive, Mark 1 attractor, BBP / prime-root ROM, Glass Key compression


In [1]:
from pathlib import Path
import math
import os
import json
import hashlib
import random
import statistics

import fitz
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display, Markdown, Image, HTML

pio.renderers.default = "notebook_connected"

BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / "Mapping the Nodes of Universal Nouns Through Recursive Computational Verbs.pdf"
OUT_DIR = BASE_DIR / "mapping_nodes_outputs"
OUT_DIR.mkdir(exist_ok=True)

print(f"PDF exists: {PDF_PATH.exists()}")
print(f"Output directory: {OUT_DIR.resolve()}")

RuntimeError: Directory 'static/' does not exist

## 1. Source metadata and rendered page gallery

The first pass renders a compact gallery of selected pages used throughout the notebook. This supports visual checking of the source layout, tables, and section structure.

In [1]:
SOURCE_PAGES = [1, 3, 4, 6, 7, 9, 11, 12, 13, 14, 15, 16]

doc = fitz.open(PDF_PATH)
metadata = {
    "page_count": doc.page_count,
    "title": doc.metadata.get("title"),
    "author": doc.metadata.get("author"),
    "subject": doc.metadata.get("subject"),
    "creator": doc.metadata.get("creator"),
    "producer": doc.metadata.get("producer"),
}
display(pd.DataFrame([metadata]))

thumb_paths = []
for page_num in SOURCE_PAGES:
    page = doc[page_num - 1]
    pix = page.get_pixmap(matrix=fitz.Matrix(1.1, 1.1), alpha=False)
    out_path = OUT_DIR / f"page_{page_num:02d}.png"
    pix.save(out_path)
    thumb_paths.append(out_path)

from PIL import Image as PILImage, ImageDraw, ImageOps

thumb_images = []
for page_num, path in zip(SOURCE_PAGES, thumb_paths):
    img = PILImage.open(path).convert("RGB")
    img.thumbnail((240, 340))
    canvas = PILImage.new("RGB", (260, 380), "white")
    draw = ImageDraw.Draw(canvas)
    canvas.paste(img, ((260 - img.width)//2, 20))
    draw.text((12, 350), f"Page {page_num}", fill="black")
    thumb_images.append(canvas)

cols = 3
rows = math.ceil(len(thumb_images)/cols)
sheet = PILImage.new("RGB", (cols*260, rows*380), "#f7f7f7")
for idx, img in enumerate(thumb_images):
    x = (idx % cols) * 260
    y = (idx // cols) * 380
    sheet.paste(img, (x, y))

gallery_path = OUT_DIR / "source_page_gallery.png"
sheet.save(gallery_path)
display(Image(filename=str(gallery_path)))

NameError: name 'fitz' is not defined

## 2. Machine-readable maps from the paper

### 2.1 Universal Component Map (pages 3-5)

The paper defines a nine-variable stack projection:

\[
\Pi(D) = (S, B, G, R, C, K, X, P, V)
\]

The table below normalizes the page-spanning source table into a structured dataframe.

In [2]:
component_rows = [
    {
        "Identifier": "S (State)",
        "Verb": "Bearing",
        "Universal Definition": "Foundational substrate capable of holding distinguishable states.",
        "Software (SHA-256 toroid)": "8x32-bit register state [a..h]",
        "Hardware (CMOS / Silicon)": "Charge distribution on the gate",
    },
    {
        "Identifier": "B (Bias)",
        "Verb": "Biasing",
        "Universal Definition": "Directed potential, asymmetry, or external field driving the operation.",
        "Software (SHA-256 toroid)": "Universal ROM constants K[i] + external message W[i]",
        "Hardware (CMOS / Silicon)": "Vdda rail voltage",
    },
    {
        "Identifier": "G (Gate)",
        "Verb": "Ruling",
        "Universal Definition": "Admissibility rule, activation barrier, or threshold condition for transition.",
        "Software (SHA-256 toroid)": "Always open; the Sziklai coupling acts natively as the gate",
        "Hardware (CMOS / Silicon)": "Threshold Vt crossing",
    },
    {
        "Identifier": "R (Route)",
        "Verb": "Routing",
        "Universal Definition": "Transport path through which data, charge, or state change propagates.",
        "Software (SHA-256 toroid)": "T1 south bridge path",
        "Hardware (CMOS / Silicon)": "NMOS/PMOS logical channel",
    },
    {
        "Identifier": "C (Coupling)",
        "Verb": "Conserving",
        "Universal Definition": "Conserved bridge that propagates state continuity and transfers influence across steps.",
        "Software (SHA-256 toroid)": "T1 shared emitter / carry-bit arithmetic overflow",
        "Hardware (CMOS / Silicon)": "Current continuity (charge in exactly equals charge out)",
    },
    {
        "Identifier": "K (Keep)",
        "Verb": "Retaining",
        "Universal Definition": "Geometric or mechanical method by which a newly generated state persists.",
        "Software (SHA-256 toroid)": "8-word continuous register shift",
        "Hardware (CMOS / Silicon)": "Output node electrical capacitance",
    },
    {
        "Identifier": "X (Address)",
        "Verb": "Addressing",
        "Universal Definition": "Selection coordinate or index of the operation within a retained sequence.",
        "Software (SHA-256 toroid)": "Discrete round index i in [0, 63]",
        "Hardware (CMOS / Silicon)": "Netlist node architectural address",
    },
    {
        "Identifier": "P (Projection)",
        "Verb": "Projecting",
        "Universal Definition": "Rendered, observable logic state, phenotype, or measurable spectrum.",
        "Software (SHA-256 toroid)": "Generated Sziklai output pairings",
        "Hardware (CMOS / Silicon)": "Verifiable logic level 0 or 1",
    },
    {
        "Identifier": "V (Verify)",
        "Verb": "Closing",
        "Universal Definition": "Lawful-fit closure test ensuring non-metastability and geometric parity.",
        "Software (SHA-256 toroid)": "Differential invariant a[i+1] - e[i+1] ≡ T2[i] - d[i] (mod 2^32)",
        "Hardware (CMOS / Silicon)": "Not actively residing in a metastable region",
    },
]
component_df = pd.DataFrame(component_rows)
component_df.to_csv(OUT_DIR / "universal_component_map.csv", index=False)
display(component_df)

NameError: name 'pd' is not defined

In [3]:
fig_component_table = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=list(component_df.columns),
                fill_color="#1f3c88",
                font=dict(color="white", size=12),
                align="left",
                height=34,
            ),
            cells=dict(
                values=[component_df[c] for c in component_df.columns],
                fill_color=[["#f7f9fc"]*len(component_df)],
                align="left",
                font=dict(size=11),
                height=38,
            ),
        )
    ]
)
fig_component_table.update_layout(
    title="Universal Component Map",
    margin=dict(l=10, r=10, t=40, b=10),
    height=540
)
fig_component_table.show()

sankey_links = []
labels = []
node_index = {}

def get_idx(label):
    if label not in node_index:
        node_index[label] = len(labels)
        labels.append(label)
    return node_index[label]

for _, row in component_df.iterrows():
    ident = row["Identifier"]
    verb = row["Verb"]
    soft = row["Software (SHA-256 toroid)"]
    hard = row["Hardware (CMOS / Silicon)"]
    for src, dst in [(ident, verb), (verb, soft), (verb, hard)]:
        sankey_links.append((get_idx(src), get_idx(dst)))

link_df = pd.DataFrame(sankey_links, columns=["source", "target"])
link_df["value"] = 1

fig_component_sankey = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            pad=18,
            thickness=16,
            line=dict(color="rgba(0,0,0,0.25)", width=0.5),
            label=labels,
            color=["#264653" if "(" in lbl else "#2a9d8f" if lbl in component_df["Verb"].values else "#e9c46a" for lbl in labels],
        ),
        link=dict(
            source=link_df["source"],
            target=link_df["target"],
            value=link_df["value"],
            color="rgba(38,70,83,0.20)",
        ),
    )
)
fig_component_sankey.update_layout(
    title="Operational mapping from identifiers to verbs to substrate-specific endpoints",
    font=dict(size=11),
    height=700,
)
fig_component_sankey.write_html(OUT_DIR / "fig_universal_component_sankey.html", include_plotlyjs="cdn")
fig_component_sankey.show()

NameError: name 'go' is not defined

### 2.2 Cross-substrate topological crystallization (pages 9-10)

The paper's Sarrus Isomorphism table is normalized below.

In [ ]:
sarrus_rows = [
    {
        "Scientific Domain": "Mechanical Engineering",
        "Physical Substrate": "Metal linkage (6R)",
        "Inward Compaction (Fold)": "Joint articulation / inward fold",
        "Outward Extension (Branch)": "Phase shift (90 degrees)",
        "Constraint Anchor": "Physical spatial guideway",
    },
    {
        "Scientific Domain": "Digital Cryptography",
        "Physical Substrate": "Silicon (SHA-256)",
        "Inward Compaction (Fold)": "Majority function Maj",
        "Outward Extension (Branch)": "Choice function Ch",
        "Constraint Anchor": "Prime K-constants / prime-roots",
    },
    {
        "Scientific Domain": "Biological Kinematics",
        "Physical Substrate": "Carbon (proteins)",
        "Inward Compaction (Fold)": "Alpha-helix formulation",
        "Outward Extension (Branch)": "Beta-sheet formulation",
        "Constraint Anchor": "Hydrophobic amino acids",
    },
]
sarrus_df = pd.DataFrame(sarrus_rows)
sarrus_df.to_csv(OUT_DIR / "sarrus_cross_substrate_map.csv", index=False)
display(sarrus_df)

fig_sarrus_table = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=list(sarrus_df.columns),
                fill_color="#4c956c",
                font=dict(color="white", size=12),
                align="left",
                height=34,
            ),
            cells=dict(
                values=[sarrus_df[c] for c in sarrus_df.columns],
                fill_color=[["#fbfffe"]*len(sarrus_df)],
                align="left",
                font=dict(size=11),
                height=42,
            ),
        )
    ]
)
fig_sarrus_table.update_layout(title="Cross-substrate topological crystallization", height=320, margin=dict(l=10, r=10, t=40, b=10))
fig_sarrus_table.show()

In [ ]:
flow_labels = []
flow_index = {}
def flow_idx(label):
    if label not in flow_index:
        flow_index[label] = len(flow_labels)
        flow_labels.append(label)
    return flow_index[label]

flow_edges = []
for _, row in sarrus_df.iterrows():
    dom = row["Scientific Domain"]
    sub = row["Physical Substrate"]
    fold = row["Inward Compaction (Fold)"]
    branch = row["Outward Extension (Branch)"]
    anchor = row["Constraint Anchor"]
    for src, dst in [(dom, sub), (sub, fold), (sub, branch), (sub, anchor)]:
        flow_edges.append((flow_idx(src), flow_idx(dst)))
flow_df = pd.DataFrame(flow_edges, columns=["source", "target"])
flow_df["value"] = 1

fig_sarrus_sankey = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=flow_labels,
            pad=18,
            thickness=16,
            line=dict(color="rgba(0,0,0,0.2)", width=0.5),
            color=["#264653" if "Engineering" in l or "Cryptography" in l or "Kinematics" in l else "#e76f51" if "(" in l else "#2a9d8f" for l in flow_labels]
        ),
        link=dict(
            source=flow_df["source"],
            target=flow_df["target"],
            value=flow_df["value"],
            color="rgba(231,111,81,0.20)"
        )
    )
)
fig_sarrus_sankey.update_layout(title="Sarrus Isomorphism: fold / branch / anchor mapping across substrates", height=600)
fig_sarrus_sankey.write_html(OUT_DIR / "fig_sarrus_cross_substrate_sankey.html", include_plotlyjs="cdn")
fig_sarrus_sankey.show()

## 3. SHA-256 bridge architecture and executable verification

Pages 6-8 of the paper describe the SHA-256 round function as a bridge-separated architecture with a North Bridge, a South Bridge, and a Sziklai coupling identity. The cells below instrument a one-block SHA-256 round engine and verify the claims that are directly testable from the specification.

### Claims verified here
- **Notebook-derived verification**: \(T2_0 = 0x08909ae5\) at the initial state.
- **Notebook-derived verification**: the differential invariant
  \[
  a_{i+1} - e_{i+1} \equiv T2_i - d_i \pmod{{2^{32}}}
  \]
  holds exactly for the instrumented round engine.
- **Notebook-derived verification**: the SHA-256 initial state and round constants are recovered exactly from prime square-root and cube-root fractional parts.


In [ ]:
MASK32 = 0xFFFFFFFF

def rotr(x, n):
    return ((x >> n) | ((x << (32 - n)) & MASK32)) & MASK32

def Ch(x, y, z):
    return (x & y) ^ (~x & z) & MASK32

def Maj(x, y, z):
    return (x & y) ^ (x & z) ^ (y & z)

def Sigma0(x):
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x):
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x):
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def sigma1(x):
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def pad_one_block(message: bytes) -> bytes:
    if len(message) > 55:
        raise ValueError("Single-block notebook implementation requires len(message) <= 55 bytes.")
    ml = len(message) * 8
    padded = message + b"\x80"
    while len(padded) % 64 != 56:
        padded += b"\x00"
    padded += ml.to_bytes(8, "big")
    return padded

def block_to_words(block: bytes):
    return [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

def schedule_from_block(block: bytes):
    W = block_to_words(block) + [0] * 48
    for t in range(16, 64):
        W[t] = (sigma1(W[t-2]) + W[t-7] + sigma0(W[t-15]) + W[t-16]) & MASK32
    return W

# Generate H0 and K from prime roots
import mpmath as mp
mp.mp.dps = 100

def first_primes(n):
    primes = []
    candidate = 2
    while len(primes) < n:
        is_prime = True
        for p in range(2, int(candidate**0.5)+1):
            if candidate % p == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(candidate)
        candidate += 1
    return primes

PRIMES = first_primes(64)
H0 = [int(mp.floor((mp.sqrt(p) - mp.floor(mp.sqrt(p))) * (2**32))) for p in PRIMES[:8]]
K = [int(mp.floor((mp.nthroot(p, 3) - mp.floor(mp.nthroot(p, 3))) * (2**32))) for p in PRIMES[:64]]

H0_hex = [f"0x{x:08x}" for x in H0]
K_hex = [f"0x{x:08x}" for x in K[:8]]
H0_hex[:4], K_hex[:4]

In [ ]:
def compress_trace_from_schedule(W, initial_state=None):
    if initial_state is None:
        initial_state = list(H0)
    a,b,c,d,e,f,g,h = initial_state
    rows = []
    for i in range(64):
        t1_sum = h + Sigma1(e) + Ch(e, f, g) + K[i] + W[i]
        t2_sum = Sigma0(a) + Maj(a, b, c)
        T1 = t1_sum & MASK32
        T2 = t2_sum & MASK32
        carry_t1 = t1_sum >> 32
        carry_t2 = t2_sum >> 32
        new_a = (T1 + T2) & MASK32
        new_e = (d + T1) & MASK32

        rows.append({
            "round": i,
            "a": a, "b": b, "c": c, "d": d, "e": e, "f": f, "g": g, "h": h,
            "W": W[i],
            "K": K[i],
            "T1": T1,
            "T2": T2,
            "carry_t1": carry_t1,
            "carry_t2": carry_t2,
            "new_a": new_a,
            "new_e": new_e,
            "lhs_diff": (new_a - new_e) & MASK32,
            "rhs_diff": (T2 - d) & MASK32,
            "invariant_ok": ((new_a - new_e) & MASK32) == ((T2 - d) & MASK32),
        })
        a, b, c, d, e, f, g, h = new_a, a, b, c, new_e, e, f, g
    return pd.DataFrame(rows)

abc_block = pad_one_block(b"abc")
abc_schedule = schedule_from_block(abc_block)
trace_abc = compress_trace_from_schedule(abc_schedule)
trace_abc.to_csv(OUT_DIR / "sha_round_trace_abc.csv", index=False)

nop_schedule = [0] * 64
trace_nop = compress_trace_from_schedule(nop_schedule)
trace_nop.to_csv(OUT_DIR / "nop_backbone_trace.csv", index=False)

ground_T2 = int(trace_nop.loc[0, "T2"])
ground_T2_hex = f"0x{ground_T2:08x}"
invariant_violations_abc = int((~trace_abc["invariant_ok"]).sum())
invariant_violations_nop = int((~trace_nop["invariant_ok"]).sum())

summary = pd.DataFrame([
    {
        "metric": "T2[0] on NOP backbone",
        "value": ground_T2_hex,
    },
    {
        "metric": "Invariant violations (abc)",
        "value": invariant_violations_abc,
    },
    {
        "metric": "Invariant violations (NOP)",
        "value": invariant_violations_nop,
    },
])
display(summary)

In [ ]:
audit_rows = []
known_H0 = [0x6a09e667,0xbb67ae85,0x3c6ef372,0xa54ff53a,0x510e527f,0x9b05688c,0x1f83d9ab,0x5be0cd19]
for idx, (expected, actual, p) in enumerate(zip(known_H0, H0, PRIMES[:8])):
    audit_rows.append({
        "type": "H0 sqrt(prime)",
        "index": idx,
        "prime": p,
        "expected_hex": f"0x{expected:08x}",
        "actual_hex": f"0x{actual:08x}",
        "match": expected == actual,
    })
known_K_first8 = [0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5]
for idx, (expected, actual, p) in enumerate(zip(known_K_first8, K[:8], PRIMES[:8])):
    audit_rows.append({
        "type": "K cbrt(prime)",
        "index": idx,
        "prime": p,
        "expected_hex": f"0x{expected:08x}",
        "actual_hex": f"0x{actual:08x}",
        "match": expected == actual,
    })
audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_DIR / "prime_root_constant_audit.csv", index=False)
display(audit_df)
assert ground_T2_hex == "0x08909ae5"
assert invariant_violations_abc == 0
assert invariant_violations_nop == 0
assert audit_df["match"].all()

In [ ]:
# A compact structural diagram for the bridge-separated round engine
node_positions = {
    "a,b,c": (0.08, 0.72),
    "e,f,g,h": (0.08, 0.28),
    "K[i], W[i]": (0.08, 0.10),
    "T2": (0.42, 0.72),
    "T1": (0.42, 0.22),
    "new_a = T1 + T2": (0.80, 0.58),
    "new_e = d + T1": (0.80, 0.18),
}

edge_list = [
    ("a,b,c", "T2"),
    ("e,f,g,h", "T1"),
    ("K[i], W[i]", "T1"),
    ("T2", "new_a = T1 + T2"),
    ("T1", "new_a = T1 + T2"),
    ("T1", "new_e = d + T1"),
]

fig_bridge = go.Figure()
for src, dst in edge_list:
    x0, y0 = node_positions[src]
    x1, y1 = node_positions[dst]
    fig_bridge.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1],
        mode="lines",
        line=dict(color="rgba(31,60,136,0.55)", width=3),
        hoverinfo="skip",
        showlegend=False
    ))

for label, (x, y) in node_positions.items():
    fig_bridge.add_trace(go.Scatter(
        x=[x], y=[y],
        mode="markers+text",
        marker=dict(size=28, color="#1f3c88" if "T" in label else "#2a9d8f"),
        text=[label],
        textposition="middle center",
        hoverinfo="text",
        showlegend=False
    ))

fig_bridge.add_annotation(
    x=0.60, y=0.39,
    text="Sziklai differential invariant: new_a - new_e ≡ T2 - d (mod 2^32)",
    showarrow=False,
    font=dict(size=12, color="#222"),
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="rgba(0,0,0,0.15)"
)
fig_bridge.update_xaxes(visible=False, range=[0, 1])
fig_bridge.update_yaxes(visible=False, range=[0, 1])
fig_bridge.update_layout(
    title="SHA-256 round architecture: North Bridge, South Bridge, and Sziklai coupling",
    height=500,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=10, r=10, t=45, b=10),
)
fig_bridge.write_html(OUT_DIR / "fig_sha_bridge_architecture.html", include_plotlyjs="cdn")
fig_bridge.show()

### 3.1 Random-message carry instrumentation

The paper treats the carry channel as a structurally important companion to the value channel. The next cells compute **notebook-derived** carry distributions over random one-block messages and summarize them by round.

In [ ]:
def random_one_block_message():
    n = random.randint(1, 55)
    return os.urandom(n)

def shannon_entropy(values):
    counts = pd.Series(values).value_counts(normalize=True)
    return float(-(counts * np.log2(counts)).sum())

random.seed(7)
np.random.seed(7)

sample_count = 256
carry_rows = []
for msg_id in range(sample_count):
    msg = random_one_block_message()
    sched = schedule_from_block(pad_one_block(msg))
    tr = compress_trace_from_schedule(sched)
    tr = tr[["round", "carry_t1", "carry_t2"]].copy()
    tr["message_id"] = msg_id
    carry_rows.append(tr)
carry_df = pd.concat(carry_rows, ignore_index=True)

carry_summary = (
    carry_df.groupby("round")
    .agg(
        mean_carry_t1=("carry_t1", "mean"),
        mean_carry_t2=("carry_t2", "mean"),
        entropy_carry_t1=("carry_t1", shannon_entropy),
        entropy_carry_t2=("carry_t2", shannon_entropy),
    )
    .reset_index()
)
carry_summary.to_csv(OUT_DIR / "random_message_carry_summary.csv", index=False)
display(carry_summary.head())

In [ ]:
fig_carry_entropy = go.Figure()
fig_carry_entropy.add_trace(go.Scatter(
    x=carry_summary["round"], y=carry_summary["entropy_carry_t1"],
    mode="lines+markers", name="T1 carry entropy", line=dict(color="#c1121f", width=3)
))
fig_carry_entropy.add_trace(go.Scatter(
    x=carry_summary["round"], y=carry_summary["entropy_carry_t2"],
    mode="lines+markers", name="T2 carry entropy", line=dict(color="#003049", width=3)
))
fig_carry_entropy.add_vline(x=7, line_dash="dash", line_color="gray", annotation_text="paper's hardness-wall marker")
fig_carry_entropy.update_layout(
    title="Notebook-derived carry entropy by round over random one-block messages",
    xaxis_title="Round",
    yaxis_title="Shannon entropy (bits)",
    height=460,
    template="plotly_white",
)
fig_carry_entropy.write_html(OUT_DIR / "fig_carry_entropy_rounds.html", include_plotlyjs="cdn")
fig_carry_entropy.show()

hist_t1 = carry_df["carry_t1"].value_counts().sort_index().reset_index()
hist_t1.columns = ["carry_t1", "count"]
fig_carry_hist = px.bar(hist_t1, x="carry_t1", y="count", title="Overall T1 carry-state counts across the random-message ensemble")
fig_carry_hist.update_layout(template="plotly_white", height=360)
fig_carry_hist.write_html(OUT_DIR / "fig_carry_t1_histogram.html", include_plotlyjs="cdn")
fig_carry_hist.show()

## 4. Stroboscopic render cycle and the 33 Hz primitive

Pages 11-12 describe a two-phase 33 Hz hardware primitive:
- **Alive phase**: 16.5 Hz active render window
- **Dead phase**: 16.5 Hz collapse / retention window
- Each half-cycle is approximately **15.15 ms**
- The compressed retained residue is stated as **896 bits = 112 bytes**

The following figure visualizes the timing structure described in the paper.

In [ ]:
frequency_hz = 33.0
period_s = 1.0 / frequency_hz
half_period_ms = 1000 * period_s / 2

t = np.linspace(0, 4 * period_s, 1200)
phase = np.sin(2 * np.pi * frequency_hz * t)
state = np.where(phase >= 0, 1, 0)

cycle_df = pd.DataFrame({
    "time_ms": t * 1000,
    "phase_signal": phase,
    "state": state,
})

fig_cycle = make_subplots(specs=[[{"secondary_y": True}]])
fig_cycle.add_trace(
    go.Scatter(x=cycle_df["time_ms"], y=cycle_df["phase_signal"], mode="lines", name="33 Hz carrier", line=dict(color="#264653", width=2)),
    secondary_y=False,
)
fig_cycle.add_trace(
    go.Scatter(x=cycle_df["time_ms"], y=cycle_df["state"], mode="lines", name="Alive / Dead state", line=dict(color="#e76f51", width=3, shape="hv")),
    secondary_y=True,
)
for k in range(4):
    start = k * period_s * 1000
    fig_cycle.add_vrect(x0=start, x1=start + half_period_ms, fillcolor="rgba(42,157,143,0.18)", line_width=0, annotation_text="Alive", annotation_position="top left")
    fig_cycle.add_vrect(x0=start + half_period_ms, x1=start + 2 * half_period_ms, fillcolor="rgba(231,111,81,0.15)", line_width=0, annotation_text="Dead", annotation_position="top left")

fig_cycle.update_layout(
    title=f"Paper-stated 33 Hz render cycle with 15.15 ms half-phases (computed half-period = {half_period_ms:.2f} ms)",
    xaxis_title="Time (ms)",
    height=470,
    template="plotly_white",
)
fig_cycle.update_yaxes(title_text="Carrier amplitude", secondary_y=False)
fig_cycle.update_yaxes(title_text="Binary phase state", range=[-0.1, 1.1], secondary_y=True)
fig_cycle.write_html(OUT_DIR / "fig_render_cycle_33hz.html", include_plotlyjs="cdn")
fig_cycle.show()

compression_df = pd.DataFrame({
    "Case": ["Rendered state", "Collapsed residue"],
    "Bits": [8 * 1024**3, 896],  # 1 GiB for scale visualization
})
fig_residue = px.bar(
    compression_df,
    x="Case",
    y="Bits",
    log_y=True,
    title="Illustrative scale contrast between rendered-state magnitude and 896-bit retained residue",
)
fig_residue.update_layout(template="plotly_white", height=360)
fig_residue.write_html(OUT_DIR / "fig_rendered_vs_residue_bits.html", include_plotlyjs="cdn")
fig_residue.show()

## 5. Mark 1 attractor and phase partition

Pages 12-13 define the Mark 1 attractor as

\[
H = \frac{\pi}{9} \approx 0.34906585
\]

and interpret it as an approximate **35% / 65%** structural partition. The cells below compute the angle, the complement, and a polar-sector view of the partition.

In [ ]:
H = math.pi / 9
partition_df = pd.DataFrame([
    {"Quantity": "H = pi / 9", "Value": H},
    {"Quantity": "H in degrees", "Value": math.degrees(H)},
    {"Quantity": "Complement (1 - H)", "Value": 1 - H},
    {"Quantity": "35% reference", "Value": 0.35},
    {"Quantity": "Difference H - 0.35", "Value": H - 0.35},
])
partition_df.to_csv(OUT_DIR / "mark1_attractor_summary.csv", index=False)
display(partition_df)

theta = np.linspace(0, 2*np.pi, 720)
fig_mark1 = go.Figure()
fig_mark1.add_trace(go.Scatterpolar(r=[1]*len(theta), theta=np.degrees(theta), mode="lines", line=dict(color="rgba(0,0,0,0.15)"), showlegend=False))
fig_mark1.add_trace(go.Barpolar(
    r=[1],
    theta=[math.degrees(H)/2],
    width=[math.degrees(H)],
    marker_color="#2a9d8f",
    opacity=0.75,
    name="H sector",
))
fig_mark1.add_trace(go.Barpolar(
    r=[1],
    theta=[math.degrees(H) + (360-math.degrees(H))/2],
    width=[360-math.degrees(H)],
    marker_color="#e9c46a",
    opacity=0.45,
    name="Complement",
))
fig_mark1.update_layout(
    title="Mark 1 attractor as an angular sector on the unit circle",
    template="plotly_white",
    polar=dict(radialaxis=dict(visible=False), angularaxis=dict(direction="clockwise", rotation=90)),
    height=500,
)
fig_mark1.write_html(OUT_DIR / "fig_mark1_polar.html", include_plotlyjs="cdn")
fig_mark1.show()

## 6. Prime-root ROM and non-sequential access

Pages 13-15 treat constants as executable boundary classes and reference both prime-root-derived constants and BBP-based direct hexadecimal access to \(\pi\).

### 6.1 Prime-root constant audit
This section uses the exact square-root and cube-root constructions that define the SHA-256 initial state and round constants.

### 6.2 BBP direct hex-word access
A direct BBP digit extractor is used below to recover 8-hex-digit words at selected offsets without computing earlier hexadecimal digits explicitly.

In [ ]:
prime_df = pd.DataFrame({
    "index": np.arange(64),
    "prime": PRIMES,
    "K_fractional": [k / 2**32 for k in K],
})
fig_prime_rom = px.scatter(
    prime_df,
    x="index",
    y="K_fractional",
    hover_data=["prime"],
    title="Prime-root-derived SHA-256 K constants normalized to fractional [0,1) values",
)
fig_prime_rom.update_traces(marker=dict(size=9, color="#1d3557"))
fig_prime_rom.update_layout(template="plotly_white", height=420, xaxis_title="Prime index", yaxis_title="Fractional constant value")
fig_prime_rom.write_html(OUT_DIR / "fig_prime_root_rom.html", include_plotlyjs="cdn")
fig_prime_rom.show()

def bbp_pi_hex_digit(n):
    n = int(n)
    def S(j, n):
        s = 0.0
        for k in range(n + 1):
            r = 8 * k + j
            s = (s + pow(16, n - k, r) / r) % 1.0
        t = 0.0
        k = n + 1
        while True:
            new_t = t + (16.0 ** (n - k)) / (8 * k + j)
            if new_t == t:
                break
            t = new_t
            k += 1
        return s + t
    x = 4 * S(1, n) - 2 * S(4, n) - S(5, n) - S(6, n)
    x = x - math.floor(x)
    return int(x * 16)

def bbp_pi_hex_word(offset):
    return "".join(f"{bbp_pi_hex_digit(offset + i):x}" for i in range(8))

offsets = [0, 8, 16, 32, 64, 128]
bbp_df = pd.DataFrame({
    "offset_hex_digits": offsets,
    "word_hex": [bbp_pi_hex_word(o) for o in offsets],
})
bbp_df.to_csv(OUT_DIR / "bbp_hex_words.csv", index=False)
display(bbp_df)

fig_bbp = px.bar(
    bbp_df,
    x="offset_hex_digits",
    y=[int(x, 16) for x in bbp_df["word_hex"]],
    title="BBP-extracted 8-hex-digit words at selected offsets",
    labels={"x": "Offset (hex digits)", "y": "32-bit word value"},
)
fig_bbp.update_layout(template="plotly_white", height=400, showlegend=False)
fig_bbp.write_html(OUT_DIR / "fig_bbp_words.html", include_plotlyjs="cdn")
fig_bbp.show()

## 7. Compression figures referenced by the paper

Page 16 states two compression-scale examples:
- **1 gigabyte** compressed to **112 bytes** (approximately **9,000,000:1** in the paper narrative)
- **40 million bits** of active human gene data compressed to **896 bits** (approximately **40,000:1** in the paper narrative)

The notebook visualizes the ratios directly from the stated source numbers.

In [ ]:
compression_cases = pd.DataFrame([
    {"Case": "1 GB -> 112 bytes", "Input_bits": 8_000_000_000, "Output_bits": 896},
    {"Case": "40,000,000 bits -> 896 bits", "Input_bits": 40_000_000, "Output_bits": 896},
])
compression_cases["Compression_ratio"] = compression_cases["Input_bits"] / compression_cases["Output_bits"]
compression_cases.to_csv(OUT_DIR / "compression_cases.csv", index=False)
display(compression_cases)

fig_compression = px.bar(
    compression_cases,
    x="Case",
    y="Compression_ratio",
    log_y=True,
    text="Compression_ratio",
    title="Paper-stated compression ratios expressed from the stated input and output sizes",
)
fig_compression.update_traces(texttemplate="%{text:.2e}", textposition="outside")
fig_compression.update_layout(template="plotly_white", height=420, yaxis_title="Compression ratio (log scale)")
fig_compression.write_html(OUT_DIR / "fig_compression_ratios.html", include_plotlyjs="cdn")
fig_compression.show()

## 8. Export summary

The notebook exports reusable datasets and standalone HTML figures to `mapping_nodes_outputs/`. This makes the notebook suitable for both rerun analysis and external presentation workflows.

In [ ]:
export_manifest = sorted([p.name for p in OUT_DIR.iterdir()])
export_df = pd.DataFrame({"exported_file": export_manifest})
display(export_df)

with open(OUT_DIR / "export_manifest.json", "w", encoding="utf-8") as f:
    json.dump(export_manifest, f, indent=2)